# road-damage-shift — Semaine 3 : Q2, robustesse au flou de bougé

Trois approches, comparées sur **les mêmes images** floutées à 5 niveaux (0 = net, puis 9, 15, 21, 31 px) :
- **(a) aucun traitement** : le modèle S1 (YOLO11s, seed 42) détecte directement sur les images floutées ;
- **(b) défloutage de Wiener** avant détection, avec le noyau exact du flou (borne optimiste : en réel il faudrait l'estimer) ;
- **(c) augmentation par flou** : même recette que S1, en ajoutant un flou de bougé aléatoire aux augmentations d'Ultralytics.

Évalué sur le test source et sur le test États-Unis, avec IC bootstrap et écarts appariés.
Protocole fixé avant les résultats : section `blur` de `configs/default.yaml`.

**Réglages Kaggle** : GPU T4 x1, Internet On, *Add Input* = le dataset des poids S1 (`yolo11s_best.pt`).
`SMOKE = True` d'abord (≈ 15 min) ; puis `SMOKE = False` via *Save Version* (≈ 2 h : l'entraînement de (c) dure comme S1).

In [ ]:
# ---- Paramètres -------------------------------------------------------------
REPO_URL = "https://github.com/<TON_USER>/road-damage-shift.git"  # à remplacer
REPO_REF = "main"
REPO_DIR = "/tmp/road-damage-shift"
RAW = "/tmp/rdd_raw"
DATA_OUT = "/tmp/rdd_yolo"
SMOKE = True                 # True : flou 0 et 15 px seulement, (c) entraîné 1 epoch sur 5 % ; False : protocole complet
S1_MANIFEST_SHA256 = "aff1da5af0e1c61885e4a59ddc058d8f659638117f99ab5d1f5df525cc57e2da"
WEIGHTS = "/kaggle/input/datasets/<TON_USER>/rds-s1-weights/yolo11s_best.pt"  # poids S1 (fichier ou dossier)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf {REPO_DIR} && git clone -q --depth 1 -b {REPO_REF} {REPO_URL} {REPO_DIR}
!pip install -q -e "{REPO_DIR}[train]" faster-coco-eval

In [ ]:
import hashlib, json, os, platform, re, subprocess, sys, time
from pathlib import Path

sys.path.insert(0, f"{REPO_DIR}/src")
import albumentations as A
import mlflow, pandas as pd, torch, ultralytics, yaml
from ultralytics import YOLO, settings

from roadshift.blur.report import blur_report, plot_blur

CFG = yaml.safe_load(open(f"{REPO_DIR}/configs/default.yaml"))
BLUR, TRAIN, CLASSES = CFG["blur"], CFG["train"], CFG["data"]["classes"]
LENGTHS = [0, 15] if SMOKE else BLUR["kernel_lengths"]
GIT_SHA = subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], text=True).strip()
VERSIONS = {"torch": torch.__version__, "ultralytics": ultralytics.__version__, "albumentations": A.__version__,
            "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}
print(GIT_SHA[:10], VERSIONS)
assert torch.cuda.is_available(), "Active le GPU T4 dans les réglages du notebook"

W = Path(WEIGHTS)
if W.is_dir():
    pts = sorted(W.rglob("*.pt")); assert len(pts) == 1, f"{W} contient {len(pts)} .pt"; W = pts[0]
W_SHA = hashlib.sha256(W.read_bytes()).hexdigest()
print("poids S1 :", W, W_SHA[:12])

## 0–1. Données (mêmes découpages qu'en S1, vérifiés par empreinte)

In [ ]:
!python -m roadshift.data.download --out {RAW} --countries Japan India Czech United_States
!python -m roadshift.data.convert --config {REPO_DIR}/configs/default.yaml --raw-root {RAW} --out {DATA_OUT} --overwrite
DATA = Path(DATA_OUT)
DATASET_INFO = json.load(open(DATA / "dataset_info.json"))
assert DATASET_INFO["manifest_sha256"] == S1_MANIFEST_SHA256, "Découpages différents de S1 : comparaison invalide"
print("Découpages identiques à S1 :", S1_MANIFEST_SHA256[:12])

## 2. Jeux de test floutés (et défloutés par Wiener)
Angle de bougé tiré une fois par image (0°, 45° ou 90°), le même à toutes les longueurs ; bruit gaussien σ = 1 % ;
Wiener avec K = 0,01 et le noyau exact. Le temps de Wiener par image (CPU) est mesuré : c'est un coût de déploiement.

In [ ]:
lengths_arg = " ".join(map(str, LENGTHS))
!python -m roadshift.blur.build --config {REPO_DIR}/configs/default.yaml --data {DATA_OUT} --workers 4 --lengths {lengths_arg}
BLUR_INFO = json.load(open(DATA / "blur_info.json"))
print({k: v for k, v in BLUR_INFO.items() if k != "config"})

## 3. MLflow

In [ ]:
def _secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None

for k in ("MLFLOW_TRACKING_USERNAME", "MLFLOW_TRACKING_PASSWORD"):
    if (v := _secret(k)):
        os.environ[k] = v
os.chdir("/kaggle/working")
mlflow.set_tracking_uri(CFG["mlflow"]["tracking_uri"] or _secret("MLFLOW_TRACKING_URI") or "sqlite:////kaggle/working/mlflow.db")
EXPERIMENT = CFG["mlflow"]["experiment"] + "-blur" + ("-smoke" if SMOKE else "")
mlflow.set_experiment(EXPERIMENT)
settings.update({"mlflow": False})
TAGS = {"stage": "S3-blur", "smoke": SMOKE, "git_sha": GIT_SHA, "dataset_sha256": DATASET_INFO["manifest_sha256"],
        "s1_weights_sha256": W_SHA, **VERSIONS}
print("expérience :", EXPERIMENT)

## 4. (c) Entraînement avec augmentation par flou
Recette S1 à l'identique (données, 50 epochs, batch, imgsz, seed 42, optimiseur auto, pas d'early stopping).
Seul changement : un `MotionBlur` (p = 0,5, longueur 3 à 31 px, angle quelconque) **ajouté** aux augmentations
Albumentations par défaut d'Ultralytics, recopiées telles quelles (8.4.x, `ultralytics/data/augment.py`).

In [ ]:
assert ultralytics.__version__.startswith("8.4"), "liste d'augmentations par défaut recopiée depuis Ultralytics 8.4.x"
AUGS = [  # défauts d'Ultralytics 8.4 (inchangés) ...
    A.Blur(p=0.01), A.MedianBlur(p=0.01), A.ToGray(p=0.01), A.CLAHE(p=0.01),
    A.RandomBrightnessContrast(p=0.0), A.RandomGamma(p=0.0), A.ImageCompression(quality_range=(75, 100), p=0.0),
    # ... + la seule différence avec S1 :
    A.MotionBlur(blur_limit=tuple(BLUR["train_aug"]["blur_limit"]), p=BLUR["train_aug"]["p"]),
]
RUNS = Path("/kaggle/working/runs_blur")
epochs, fraction = (1, 0.05) if SMOKE else (TRAIN["epochs"], 1.0)
spec = TRAIN["models"][BLUR["model"]]
with mlflow.start_run(run_name="train_yolo11s_bluraug" + ("_smoke" if SMOKE else "")) as run:
    mlflow.set_tags({**TAGS, "method": "aug"})
    mlflow.log_params({"model": BLUR["model"], "weights": spec["weights"], "epochs": epochs, "fraction": fraction,
                       "imgsz": TRAIN["imgsz"], "batch": TRAIN["batch"], "seed": CFG["project"]["seed"],
                       "patience": TRAIN["patience"], "augmentations": repr(AUGS[-1]),
                       "motion_blur_p": BLUR["train_aug"]["p"], "motion_blur_limit": str(BLUR["train_aug"]["blur_limit"])})
    t0 = time.time()
    m = YOLO(spec["weights"])
    m.train(data=str(DATA / "data_source.yaml"), epochs=epochs, fraction=fraction, imgsz=TRAIN["imgsz"],
            batch=TRAIN["batch"], seed=CFG["project"]["seed"], deterministic=TRAIN["deterministic"],
            patience=TRAIN["patience"], workers=TRAIN["workers"], amp=TRAIN["amp"], device=0,
            augmentations=AUGS, project=str(RUNS), name="train_bluraug", exist_ok=True, plots=True)
    mlflow.log_metric("time/train_minutes", (time.time() - t0) / 60)
    save_dir = Path(m.trainer.save_dir)
    df = pd.read_csv(save_dir / "results.csv"); df.columns = [c.strip() for c in df.columns]
    for _, row in df.iterrows():
        mlflow.log_metrics({"epoch/" + re.sub(r"[^\w\-./ ]", "_", c): float(row[c])
                            for c in df.columns if c != "epoch" and pd.notna(row[c])}, step=int(row["epoch"]))
    AUG_W = save_dir / "weights" / "best.pt"
    mlflow.log_artifact(str(AUG_W), artifact_path="weights")
    mlflow.log_artifact(str(save_dir / "results.csv"), artifact_path="train")
    AUG_RUN_ID = run.info.run_id
print("modèle (c) :", AUG_W)

## 5. Évaluations : (a) et (b) avec le modèle S1, (c) avec le modèle augmenté

In [ ]:
def evaluate(weights, run_dir, tag, yml, split="val"):
    YOLO(str(weights)).val(data=str(DATA / yml), split=split, imgsz=TRAIN["imgsz"], batch=TRAIN["batch"],
                           conf=CFG["eval"]["conf_map"], save_json=True, plots=False, device=0,
                           project=str(run_dir), name=f"eval_{tag}", exist_ok=True, verbose=False)

BASE_DIR, AUG_DIR = RUNS / "eval_base", RUNS / "eval_aug"
t0 = time.time()
for weights, run_dir, kinds in ((W, BASE_DIR, ("blur", "wiener")), (AUG_W, AUG_DIR, ("blur",))):
    evaluate(weights, run_dir, "source_val", "data_source.yaml")  # seuil du F1 : validation source nette
    for s in BLUR["eval_sets"]:
        for L in LENGTHS:
            for kind in kinds:
                if kind == "wiener" and L == 0:
                    continue  # Wiener sans flou = image d'origine
                evaluate(weights, run_dir, f"{kind}_L{L}_{s}", f"data_{kind}_L{L}_{s}.yaml")
print(f"évaluations : {(time.time() - t0) / 60:.1f} min")

## 6. Résultats : IC bootstrap, écarts appariés, figure

In [ ]:
OUT = Path("/kaggle/working/results"); OUT.mkdir(exist_ok=True)
suffix = "_smoke" if SMOKE else ""
scores, gains = blur_report(BASE_DIR, AUG_DIR, DATA, n_boot=CFG["eval"]["bootstrap"]["n_resamples"])
pd.DataFrame(scores).to_csv(OUT / f"blur_scores{suffix}.csv", index=False)
pd.DataFrame(gains).to_csv(OUT / f"blur_gains{suffix}.csv", index=False)
png = plot_blur(scores, OUT / f"blur_curve{suffix}.png")
for f in ("blur_info.json", "blur_manifest.csv"):
    (OUT / f.replace(".", f"{suffix}.", 1)).write_bytes((DATA / f).read_bytes())
with mlflow.start_run(run_name="blur_summary" + suffix):
    mlflow.set_tags({**TAGS, "aug_train_run_id": AUG_RUN_ID})
    mlflow.log_metrics({"wiener_ms_median": BLUR_INFO["wiener_ms_median"], "wiener_ms_p90": BLUR_INFO["wiener_ms_p90"]})
    for r in scores:
        mlflow.log_metrics({f"{r['method']}/{r['eval_set']}/L{r['blur_length']}/{k}": float(r[k])
                            for k in ("mAP50", "mAP50_lo", "mAP50_hi", "F1", "F1_lo", "F1_hi")})
    for f in OUT.glob(f"blur_*{suffix}.*"):
        mlflow.log_artifact(str(f), artifact_path="results")
display(pd.DataFrame(scores)[["method", "eval_set", "blur_length", "mAP50", "mAP50_lo", "mAP50_hi", "F1"]])
display(pd.DataFrame(gains))
from IPython.display import Image
Image(str(png))

## 7. Récupérer les sorties (onglet *Output* de la version)

In [ ]:
!cd /kaggle/working && zip -qr s3_blur_outputs.zip results mlflow.db mlruns 2>/dev/null; ls -lh /kaggle/working

## À me renvoyer
`s3_blur_outputs.zip`. Aucune conclusion n'est tirée avant ces chiffres et leurs IC.